In [1]:
!pip install sentence-transformers faiss-cpu


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer


Load knowledge base

In [3]:
kb_path = r"C:/Users/Moetez/OneDrive/ml_tutor_project/data/ml_knowledge_base/ml_concepts.json"

with open(kb_path, "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

print("Number of knowledge chunks:", len(knowledge_base))

Number of knowledge chunks: 4


In [4]:
documents = [item["content"] for item in knowledge_base]

documents

['Linear regression is a machine learning algorithm used to predict continuous numerical values by finding the best linear relationship between variables.',
 'Overfitting happens when a machine learning model memorizes training data too closely and performs poorly on unseen data.',
 "Gradient descent is an optimization algorithm used to minimize a model's error by adjusting parameters step by step.",
 'Decision trees are models that make predictions by splitting data into branches based on feature values.']

Load embedding model

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generate embeddings

In [6]:
doc_embeddings = model.encode(documents)

doc_embeddings.shape

(4, 384)

Create FAISS index

In [7]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

print("FAISS index size:", index.ntotal)

FAISS index size: 4


Create retieval function

In [8]:
def retrieve(query, top_k=2):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(knowledge_base[idx])

    return results

Test retieval function

In [9]:
query = "Why does overfitting happen?"

results = retrieve(query)

results

[{'topic': 'overfitting',
  'difficulty': 'beginner',
  'content': 'Overfitting happens when a machine learning model memorizes training data too closely and performs poorly on unseen data.'},
 {'topic': 'gradient descent',
  'difficulty': 'beginner',
  'content': "Gradient descent is an optimization algorithm used to minimize a model's error by adjusting parameters step by step."}]

Pretty Print Results

In [10]:
for r in results:

    print("="*80)

    print("TOPIC:", r["topic"])

    print("\nCONTENT:")
    print(r["content"])

TOPIC: overfitting

CONTENT:
Overfitting happens when a machine learning model memorizes training data too closely and performs poorly on unseen data.
TOPIC: gradient descent

CONTENT:
Gradient descent is an optimization algorithm used to minimize a model's error by adjusting parameters step by step.


Add simple conversation memory

In [11]:
conversation_history = [
    "What is overfitting?",
    "Why is it bad?"
]

Build contextual query

In [12]:
current_question = "How can we prevent it?"

contextual_query = " ".join(conversation_history) + " " + current_question

print(contextual_query)

What is overfitting? Why is it bad? How can we prevent it?


Retrieve using context

In [13]:
results = retrieve(contextual_query)

for r in results:

    print("="*80)

    print(r["topic"])

    print(r["content"])

overfitting
Overfitting happens when a machine learning model memorizes training data too closely and performs poorly on unseen data.
gradient descent
Gradient descent is an optimization algorithm used to minimize a model's error by adjusting parameters step by step.
